In [ ]:
!pip install peft faiss-cpu sentence-transformers transformers torch tqdm rouge-score bert-score nltk openai google-generativeai matplotlib seaborn pyyaml

## Google Colab

In [ ]:
!git clone https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git

In [ ]:
%cd Emasters_Group-2_CapstoneProject

In [ ]:
!ls -la /content/Emasters_Group-2_CapstoneProject

In [ ]:
sys.path.append('/content/Emasters_Group-2_CapstoneProject')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/generators')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/evaluation')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/rag')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/embeddings')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/configs')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/output')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/outputs')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/scripts')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/plots')

In [1]:
%%time

import sys
import os
import json
import logging
from pathlib import Path
import yaml
import random
import time as _time
import torch
import numpy as np
import pandas as pd

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

project_root = Path.cwd()
sys.path.append(str(project_root))
sys.path.append(str(project_root / "src"))

# Import all modules
from config import load_config
from src.data.data_loader import DatasetLoader
from src.models.load_model import ModelLoader
from src.generators.program_generator import GenerationPipeline
from src.generators.doc_generator import DocGenerator
from src.generators.commit_generator import CommitMessageGenerator
from src.generators.text_to_mongo_generator import TextToMongoGenerator
from src.generators.llm_baseline import LargeLLMGenerator
from src.vectorDB.index_manager import IndexManager
from src.rag.rag_pipeline import RAGPipeline
from src.evaluation.comparator import ModelComparator
from src.evaluation.rag_evaluation import RAGEvaluator
from src.evaluation.visualization import ResultVisualizer
from src.evaluation.metrics import EvaluationMetrics

print("✅ All modules imported successfully!")

2026-08-27 20:17:17,196 - faiss.loader - INFO - Loading faiss with AVX2 support.
2026-08-27 20:17:17,197 - faiss.loader - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-08-27 20:17:17,197 - faiss.loader - INFO - Loading faiss.
2026-08-27 20:17:17,256 - faiss.loader - INFO - Successfully loaded faiss.


✅ All modules imported successfully!
CPU times: user 5.83 s, sys: 1.26 s, total: 7.09 s
Wall time: 7.12 s


In [2]:
%%time 

print("\n" + "="*60)
print("STEP 1: Loading Configuration")
print("="*60)

config = load_config("configs/config.yaml")
bertscore_model_resolved = config.evaluation.resolve_bertscore_model(base_dir=project_root)

print(f"Project: {config.model_name}")
print(f"Model: {config.model_name}")
print(f"Embedding: {config.embedding.external_model}")
print(f"Device: {config.evaluation.device}")
print(f"LoRA Output: {config.lora.output_dir}")
if bertscore_model_resolved == config.evaluation.bertscore_model:
    print(f"BERTScore backbone: {bertscore_model_resolved}  (⚠️ local checkpoint not found at "
          f"'{config.evaluation.bertscore_local_path}' -- will download from Hugging Face Hub)")
else:
    print(f"BERTScore backbone: {bertscore_model_resolved}  (✅ loaded from local checkpoint)")
print("✅ Configuration loaded successfully!")



STEP 1: Loading Configuration
Project: Qwen/Qwen2.5-Coder-0.5B-Instruct
Model: Qwen/Qwen2.5-Coder-0.5B-Instruct
Embedding: BAAI/bge-small-en-v1.5
Device: cpu
LoRA Output: models/checkpoints/lora_finetuned
BERTScore backbone: /Users/anjansuputra/Documents/GitHub/Emasters_Group-2_CapstoneProject/models/codebert-base  (✅ loaded from local checkpoint)
✅ Configuration loaded successfully!
CPU times: user 9.55 ms, sys: 1.77 ms, total: 11.3 ms
Wall time: 10.3 ms


In [3]:
%%time

print("\n" + "="*60)
print("STEP 2: Loading Dataset")
print("="*60)

data_loader = DatasetLoader(target_filename="qwen_spider_mongodb_conversion.json")
spider_data = data_loader.load_data()

print(f"✅ Loaded {len(spider_data)} samples from Spider dataset")

# Prepare training data
training_data = data_loader.get_training_data()
print(f"✅ Prepared {len(training_data)} training examples")

2026-08-27 20:17:20,007 - src.data.data_loader - INFO - ✅ Found qwen_spider_mongodb_conversion.json at: /Users/anjansuputra/Documents/GitHub/Emasters_Group-2_CapstoneProject/qwen_spider_mongodb_conversion.json
2026-08-27 20:17:20,008 - src.data.data_loader - INFO - 📂 Loading data from /Users/anjansuputra/Documents/GitHub/Emasters_Group-2_CapstoneProject/qwen_spider_mongodb_conversion.json...
2026-08-27 20:17:20,014 - src.data.data_loader - INFO - ✅ Successfully loaded 1114 records.
2026-08-27 20:17:20,015 - src.data.data_loader - INFO - ✅ Prepared 1114 training examples.



STEP 2: Loading Dataset
✅ Loaded 1114 samples from Spider dataset
✅ Prepared 1114 training examples
CPU times: user 18.1 ms, sys: 25.3 ms, total: 43.4 ms
Wall time: 68.5 ms


In [4]:
%%time 

print("\n" + "="*60)
print("STEP 3: Building Semantic & AST Indices")
print("="*60)

# Prepare documents for indexing
documents = []
for item in spider_data:
    question = item.get("question", "")
    mongodb_query = item.get("generated_mongodb_query", "")
    if isinstance(mongodb_query, list) and len(mongodb_query) > 0:
        mongodb_query = mongodb_query[0]

    documents.append({
        "text": question,
        "mongodb_query": mongodb_query,
        "generated_mongodb_query": mongodb_query,
        "question": question,
        "database_id": item.get("database_id", "")
    })

# Initialize index manager
index_manager = IndexManager(
    embedding_model=config.embedding.external_model,
    language=config.retrieval.structural_language,  # Now "mongodb"
    semantic_weight=1.0 - config.retrieval.structural_weight_alpha,
    ast_weight=config.retrieval.structural_weight_alpha,
    save_dir="data/indices",
)

index_manager.build_indices(
    documents=documents,
    text_field="question",
    code_field="mongodb_query",
    batch_size=config.embedding.batch_size,
    force_rebuild=True
)

print("✅ Indices built and saved successfully!")

2026-08-27 20:17:20,029 - src.vectorDB.index_manager - INFO - IndexManager initialized: model=BAAI/bge-small-en-v1.5, lang=mongodb
2026-08-27 20:17:20,029 - src.vectorDB.index_manager - INFO - Building indices for 1114 documents...
2026-08-27 20:17:20,030 - vectorDB.embedder - INFO - Loading embedding model: BAAI/bge-small-en-v1.5 on cpu



STEP 3: Building Semantic & AST Indices


2026-08-27 20:17:20,865 - vectorDB.embedder - INFO - ✅ Embedding model loaded successfully
2026-08-27 20:17:20,866 - src.vectorDB.index_manager - INFO - 
=== Building Semantic Index ===
Encoding texts: 100%|██████████| 35/35 [00:07<00:00,  4.68batch/s]
2026-08-27 20:17:28,364 - vectorDB.embedder - INFO - ✅ Encoded 1114 texts -> shape torch.Size([1114, 384])
2026-08-27 20:17:28,367 - vectorDB.semantic_index - INFO - Initializing FAISS index: dim=384, type=Flat
2026-08-27 20:17:28,368 - vectorDB.semantic_index - INFO - Adding 1114 vectors to FAISS index...
2026-08-27 20:17:28,370 - vectorDB.semantic_index - INFO - ✅ Index now contains 1114 vectors
2026-08-27 20:17:28,370 - src.vectorDB.index_manager - INFO - 
=== Building AST Index ===
2026-08-27 20:17:28,371 - vectorDB.ast_index - INFO - Initializing AST index for language: mongodb
Building AST index: 100%|██████████| 1114/1114 [00:00<00:00, 70541.46snippet/s]
2026-08-27 20:17:28,389 - vectorDB.ast_index - INFO - ✅ AST index built with 

✅ Indices built and saved successfully!
CPU times: user 47 s, sys: 1.84 s, total: 48.9 s
Wall time: 8.37 s


In [5]:
%%time 

print("\n" + "="*60)
print("STEP 4: LoRA Fine-Tuning Setup")
print("="*60)

# Initialize model loader
model_loader = ModelLoader(config)
tokenizer = model_loader.load_tokenizer()
base_model = model_loader.load_base_model()

# Check if LoRA model already exists
lora_path = Path(config.lora.output_dir)
if lora_path.exists() and (lora_path / "adapter_config.json").exists():
    print(f"✅ Found existing LoRA adapter at {lora_path}")
    lora_model = model_loader.autodetect_saved_model(str(lora_path))
else:
    print("🔄 Training new LoRA adapter...")
    
    # Setup LoRA
    lora_model = model_loader.setup_lora(
        trainable_fraction=config.lora.trainable_fraction,
        r=config.lora.r,
        alpha=config.lora.alpha,
        dropout=config.lora.dropout
    )
    
    # Train LoRA
    lora_model = model_loader.train_lora(
        train_data=training_data,
        output_dir=str(lora_path),
        epochs=config.lora.epochs,
        batch_size=config.lora.batch_size,
        grad_accum=4,
        lr=config.lora.learning_rate,
        max_length=config.generation.max_length
    )
    
    print(f"✅ LoRA adapter saved to {lora_path}")

# Load models for generation
models, tokenizer = model_loader.load_models(lora_path=str(lora_path))
print("✅ Models loaded successfully!")

2026-08-27 20:17:28,467 - src.models.load_model - INFO - 🖥️ Initialized ModelLoader | Device: mps | Dtype: torch.float16
2026-08-27 20:17:28,469 - src.models.load_model - INFO - 🔄 Loading tokenizer: Qwen/Qwen2.5-Coder-0.5B-Instruct



STEP 4: LoRA Fine-Tuning Setup


2026-08-27 20:17:29,086 - src.models.load_model - INFO - 🔄 Loading base model: Qwen/Qwen2.5-Coder-0.5B-Instruct
2026-08-27 20:17:32,406 - src.models.load_model - INFO - ✅ Found saved LoRA adapter at models/checkpoints/lora_finetuned. Loading...


✅ Found existing LoRA adapter at models/checkpoints/lora_finetuned


2026-08-27 20:17:33,459 - src.models.load_model - INFO - Reusing already-loaded LoRA model in memory.


✅ Models loaded successfully!
CPU times: user 9.35 s, sys: 2.63 s, total: 12 s
Wall time: 5.04 s


In [6]:
%%time 

print("\n" + "="*60)
print("STEP 5: Program Generation Tasks")
print("="*60)

# Initialize generation pipelines
base_gen_pipeline = GenerationPipeline(models["base"], tokenizer, config.generation)
lora_gen_pipeline = GenerationPipeline(models["lora"], tokenizer, config.generation)

# Task 1: Documentation Generation
print("\n[Task 1: Documentation Generation]")
doc_gen = DocGenerator(lora_gen_pipeline)
sample_code = "def calculate_area(radius):\n    return 3.14159 * radius ** 2"
docstring = doc_gen.generate_docstring(sample_code)
print(f"Code:\n{sample_code}")
print(f"\nGenerated Docstring:\n{docstring}")

# Task 2: Text-to-MongoDB Generation
print("\n[Task 2: Text-to-MongoDB Generation]")
text_to_mongo_gen = TextToMongoGenerator(lora_gen_pipeline, config)
sample_question = "How many actors do we have?"
result = text_to_mongo_gen.generate_single(sample_question)
print(f"Question: {sample_question}")
print(f"Generated MongoDB Query: {result['generated_query']}")
print(f"Latency: {result['latency_ms']:.2f} ms")

# Task 3: Commit Message Generation
print("\n[Task 3: Commit Message Generation]")
commit_gen = CommitMessageGenerator(lora_gen_pipeline)
sample_diff = "diff --git a/main.py b/main.py\n+import os\n+print('Hello World')"
commit_msg = commit_gen.generate_commit_msg(sample_diff)
print(f"Diff:\n{sample_diff}")
print(f"\nGenerated Commit Message:\n{commit_msg}")

print("\n✅ All generation tasks completed!")

/Users/anjansuputra/anaconda3/envs/ApexEnv/lib/python3.10/site-packages/transformers/generation/utils.py:1562: UserWarning: The operator 'aten::isin.Tensor_Tensor_out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:13.)
  and torch.isin(elements=eos_token_tensor, test_elements=pad_token_tensor).any()



STEP 5: Program Generation Tasks

[Task 1: Documentation Generation]
Code:
def calculate_area(radius):
    return 3.14159 * radius ** 2

Generated Docstring:
Computes the area of a circle based on its radius. The formula used is πr² where r is the radius. This function takes a single argument representing the radius of the circle and returns its calculated area.

[Task 2: Text-to-MongoDB Generation]
Question: How many actors do we have?
Generated MongoDB Query: db.actors.countDocuments()db.actor.find({}, {count: 1, _id: 0})[0].count
Latency: 69837.53 ms

[Task 3: Commit Message Generation]
Diff:
diff --git a/main.py b/main.py
+import os
+print('Hello World')

Generated Commit Message:
feat(main): Add print statement to main.py file#1234567890abcdefgghhiiijjjkkllmmnnnoopqrstuvwxz

✅ All generation tasks completed!
CPU times: user 2min 9s, sys: 18.2 s, total: 2min 27s
Wall time: 2min 27s


In [7]:
%%time 

print("\n" + "="*60)
print("STEP 6: RAG Pipeline Setup")
print("="*60)

# Initialize RAG pipeline
rag_pipeline = RAGPipeline(
    retriever=index_manager,
    generator=lora_gen_pipeline,
    config=config.generation
)

# Test RAG generation
test_query = "How many pets do we have?"
print(f"\nTest Query: {test_query}")

rag_output, retrieved_context = rag_pipeline.generate_with_rag(test_query, top_k=3)
print(f"\nRetrieved {len(retrieved_context)} context items")
print(f"\nGenerated MongoDB Query:\n{rag_output}")

print("\n✅ RAG pipeline initialized successfully!")


STEP 6: RAG Pipeline Setup

Test Query: How many pets do we have?

Retrieved 3 context items

Generated MongoDB Query:
db.pets.countDocuments()SELECT count(*) FROM pets

✅ RAG pipeline initialized successfully!
CPU times: user 7.4 s, sys: 835 ms, total: 8.24 s
Wall time: 6.65 s


In [8]:
%%time

print("\n" + "="*60)
print("STEP 7: RAG Evaluation & Top-K Sweep")
print("="*60)

# Prepare evaluation data
eval_samples = spider_data[:5]
eval_queries = [item.get("question", "") for item in eval_samples]
eval_refs = []
for item in eval_samples:
    gold = item.get("generated_mongodb_query", "")
    if isinstance(gold, list) and len(gold) > 0:
        gold = gold[0]
    eval_refs.append(gold)

baseline_generator = TextToMongoGenerator(lora_gen_pipeline, config)

print("\nGenerating baseline predictions (LoRA without RAG)...")
baseline_preds = []
for q in eval_queries:
    result = baseline_generator.generate_single(q)
    baseline_preds.append(result["generated_query"])

comparator = ModelComparator(device=config.evaluation.device, bertscore_model=bertscore_model_resolved)
rag_evaluator = RAGEvaluator(comparator)

# Measure RAG improvement
print("\nMeasuring RAG improvement...")
rag_improvement = rag_evaluator.measure_rag_improvement(
    rag_pipeline=rag_pipeline,
    references=eval_refs,
    queries=eval_queries,
    lora_preds=baseline_preds,
    top_k=3
)

print(f"\nRAG vs. {rag_improvement['baseline_used_for_gain']} (no-RAG) — gain per metric:")
for metric, gain in rag_improvement['gain_over_baseline'].items():
    arrow = "↑" if gain > 0 else ("↓" if gain < 0 else "→")
    print(f"  {metric:20s}: {gain:+.4f} {arrow}")

# ========================================
# Top-K Sweep
# ========================================
print("\n" + "="*60)
print("Top-K Retrieval Sweep")
print("="*60)

k_values = config.retrieval.topk_sweep  # [1, 2, 3, 5, 8]
sweep_results = rag_evaluator.sweep_top_k(
    rag_pipeline=rag_pipeline,
    references=eval_refs,
    queries=eval_queries,
    baseline_preds=baseline_preds,
    k_values=k_values,
    verbose=True,  # prints one aligned table covering every k in k_values (1, 2, 3, 5, 8, ...)
)

best_k = rag_evaluator.best_k(sweep_results, metric="BLEU")
print(f"\n✅ Best top_k by BLEU gain: {best_k}")

# Plot results
rag_evaluator.plot_gain_vs_k(
    sweep_results,
    save_path=str(Path(config.outputs.plots_dir) / "topk_gain_sweep.png")
)

print("\n✅ RAG evaluation completed!")


STEP 7: RAG Evaluation & Top-K Sweep

Generating baseline predictions (LoRA without RAG)...


2026-08-27 20:23:47,679 - src.evaluation.comparator - INFO - ModelComparator using BERTScore/CodeBERTScore backbone: /Users/anjansuputra/Documents/GitHub/Emasters_Group-2_CapstoneProject/models/codebert-base
2026-08-27 20:23:47,680 - src.evaluation.rag_evaluation - INFO - RAGEvaluator using BERTScore/CodeBERTScore backbone: /Users/anjansuputra/Documents/GitHub/Emasters_Group-2_CapstoneProject/models/codebert-base
2026-08-27 20:23:47,680 - src.evaluation.rag_evaluation - INFO - Measuring RAG improvement over baseline LoRA (top_k=3)...



Measuring RAG improvement...


2026-08-27 20:27:57,482 - absl - INFO - Using default tokenizer.
2026-08-27 20:27:57,485 - src.evaluation.rag_evaluation - INFO - Sweeping top_k=1...



RAG vs. Baseline_LoRA (no-RAG) — gain per metric:
  BLEU                : -0.2060 ↓
  BERTScore           : -0.0900 ↓

Top-K Retrieval Sweep


2026-08-27 20:30:12,649 - absl - INFO - Using default tokenizer.
2026-08-27 20:30:12,652 - src.evaluation.rag_evaluation - INFO - Sweeping top_k=2...
2026-08-27 20:33:46,579 - absl - INFO - Using default tokenizer.
2026-08-27 20:33:46,584 - src.evaluation.rag_evaluation - INFO - Sweeping top_k=3...
2026-08-27 20:37:31,062 - absl - INFO - Using default tokenizer.
2026-08-27 20:37:31,066 - src.evaluation.rag_evaluation - INFO - Sweeping top_k=5...
2026-08-27 20:42:10,974 - absl - INFO - Using default tokenizer.
2026-08-27 20:42:10,976 - src.evaluation.rag_evaluation - INFO - Sweeping top_k=8...
2026-08-27 20:50:23,831 - absl - INFO - Using default tokenizer.
2026-08-27 20:50:24,017 - src.evaluation.rag_evaluation - INFO - ✅ Saved Top-K sweep plot to output/plots/topk_gain_sweep.png



Top-K Retrieval Sweep Results
+-------+--------+---------+-----------+--------------+
| Top-K |  BLEU  | ROUGE-1 | BLEU Gain | ROUGE-1 Gain |
+-------+--------+---------+-----------+--------------+
|   1   | 0.0057 |  0.2694 |  -0.2043  |   -0.2300    |
|   2   | 0.0081 |  0.2468 |  -0.2019  |   -0.2526    |
|   3   | 0.0023 |  0.1864 |  -0.2077  |   -0.3130    |
|   5   | 0.0056 |  0.1802 |  -0.2044  |   -0.3192    |
|   8   | 0.0067 |  0.2089 |  -0.2033  |   -0.2905    |
+-------+--------+---------+-----------+--------------+

✅ Best top_k by BLEU gain: 2

✅ RAG evaluation completed!
CPU times: user 26min 34s, sys: 3min 4s, total: 29min 39s
Wall time: 30min 16s


In [13]:
%%time

print("\n" + "="*60)
print("STEP 8: Comprehensive Model Comparison")
print("="*60)

# Prepare evaluation data
eval_samples = spider_data[:5]
eval_queries = [item.get("question", "") for item in eval_samples]
eval_refs = []
for item in eval_samples:
    mongo_q = item.get("generated_mongodb_query", "")
    if isinstance(mongo_q, list) and len(mongo_q) > 0:
        mongo_q = mongo_q[0]
    eval_refs.append(mongo_q)

print(f"\nEvaluating {len(eval_queries)} samples")

import time as _time

base_generator = TextToMongoGenerator(base_gen_pipeline, config)
lora_generator = TextToMongoGenerator(lora_gen_pipeline, config)

# ========================================
# 1. Base model predictions
# ========================================
print("\n[1/4] Generating Base Model predictions...")
base_preds = []
base_start = _time.perf_counter()
for q in eval_queries:
    base_preds.append(base_generator.generate_single(q)["generated_query"])
base_latency = (_time.perf_counter() - base_start) / len(eval_queries) * 1000
print(f"  ✅ Done | Avg latency: {base_latency:.1f} ms/query")

# ========================================
# 2. LoRA model predictions
# ========================================
print("[2/4] Generating LoRA Model predictions...")
lora_preds = []
lora_start = _time.perf_counter()
for q in eval_queries:
    lora_preds.append(lora_generator.generate_single(q)["generated_query"])
lora_latency = (_time.perf_counter() - lora_start) / len(eval_queries) * 1000
print(f"  ✅ Done | Avg latency: {lora_latency:.1f} ms/query")

# ========================================
# 3. RAG predictions
# ========================================
print(f"[3/4] Generating RAG predictions (top_k={best_k})...")
rag_preds = []
rag_start = _time.perf_counter()
for q in eval_queries:
    pred, _ = rag_pipeline.generate_with_rag(q, top_k=best_k)
    rag_preds.append(pred)
rag_latency = (_time.perf_counter() - rag_start) / len(eval_queries) * 1000
print(f"  ✅ Done | Avg latency: {rag_latency:.1f} ms/query")

# ========================================
# 4. LLM baseline predictions
# ========================================
print("[4/4] Generating LLM Baseline predictions...")
llm_preds = None
llm_latency = None
llm_generator = None
try:
    llm_baseline = LargeLLMGenerator(config=config)
    if llm_baseline.backend == "api" and llm_baseline.client is None:
        print("  ⚠️ LLM baseline skipped: Gemini client not initialized (no/invalid API key).")
    else:
        llm_generator = TextToMongoGenerator(llm_baseline, config)
        llm_preds = []
        llm_start = _time.perf_counter()
        for q in eval_queries:
            llm_preds.append(llm_generator.generate_single(q)["generated_query"])
        llm_latency = (_time.perf_counter() - llm_start) / len(eval_queries) * 1000
        print(f"  ✅ Done | Avg latency: {llm_latency:.1f} ms/query")
except Exception as e:
    print(f"  ⚠️ LLM baseline skipped: {e}")
    llm_preds = None

# ========================================
# Calculate Evaluation Metrics
# ========================================
print("\n" + "="*60)
print("Calculating Evaluation Metrics")
print("="*60)

from src.evaluation.metrics import EvaluationMetrics
metrics = EvaluationMetrics()

def compute_all_metrics(refs, preds, device="cpu"):
    """Compute all metrics for a set of predictions."""
    result = {}
    result["BLEU"] = metrics.compute_bleu(refs, preds)

    rouge = metrics.compute_rouge(refs, preds)
    result["ROUGE-1"] = rouge.get("ROUGE-1", 0.0)
    result["ROUGE-L"] = rouge.get("ROUGE-L", 0.0)
    result["BERTScore"] = metrics.compute_bertscore(
        refs, preds, device=device, model_type=bertscore_model_resolved
    )

    # exact_matches = [metrics.compute_exact_match(r, p) for r, p in zip(refs, preds)]
    # result["Exact_Match"] = round(sum(exact_matches) / len(exact_matches), 4) if exact_matches else 0.0

    resp_acc = [metrics.compute_response_accuracy(r, p) for r, p in zip(refs, preds)]
    result["Response_Accuracy"] = round(sum(resp_acc) / len(resp_acc), 4) if resp_acc else 0.0

    return result

# Compute metrics for each model
model_results = {}
model_latencies = {
    "Base_Model": base_latency,
    "LoRA_Model": lora_latency,
    "RAG_Pipeline": rag_latency,
}
if llm_latency is not None:
    model_latencies["LLM_Baseline"] = llm_latency

print("\n  Computing Base_Model metrics...")
model_results["Base_Model"] = compute_all_metrics(eval_refs, base_preds, device=config.evaluation.device)

print("  Computing LoRA_Model metrics...")
model_results["LoRA_Model"] = compute_all_metrics(eval_refs, lora_preds, device=config.evaluation.device)

print("  Computing RAG_Pipeline metrics...")
model_results["RAG_Pipeline"] = compute_all_metrics(eval_refs, rag_preds, device=config.evaluation.device)

if llm_preds is not None:
    print("  Computing LLM_Baseline metrics...")
    model_results["LLM_Baseline"] = compute_all_metrics(eval_refs, llm_preds, device=config.evaluation.device)

# ========================================
# Print Comparison Table
# ========================================
print("\n" + "="*60)
print("📊 Model Comparison Results")
print("="*60)

metric_names = list(model_results["Base_Model"].keys())
model_names = list(model_results.keys())

# Header
header = f"{'Metric':<20}"
for m in model_names:
    header += f" {m:>14}"
print(f"\n{header}")
print("-" * (20 + 15 * len(model_names)))

# Rows
for metric in metric_names:
    row = f"{metric:<20}"
    for model in model_names:
        val = model_results[model].get(metric, 0.0)
        row += f" {val:>14.4f}"
    print(row)

# Latency row
print("-" * (20 + 15 * len(model_names)))
latency_row = f"{'Avg_Latency (ms)':<20}"
for model in model_names:
    lat = model_latencies.get(model, 0.0)
    latency_row += f" {lat:>14.1f}"
print(latency_row)

# ========================================
# Print Gain Analysis
# ========================================
print("\n" + "="*60)
print("📈 Gain Analysis (vs Base_Model)")
print("="*60)

gain_header = f"{'Metric':<20} {'LoRA Gain':>12} {'RAG Gain':>12} {'RAG vs LoRA':>12}"
if "LLM_Baseline" in model_results:
    gain_header += f" {'LLM Gain':>12}"
print(f"\n{gain_header}")
print("-" * (20 + 13 * (4 if "LLM_Baseline" in model_results else 3)))

for metric in metric_names:
    base_v = model_results["Base_Model"][metric]
    lora_v = model_results["LoRA_Model"][metric]
    rag_v = model_results["RAG_Pipeline"][metric]

    lora_gain = lora_v - base_v
    rag_gain = rag_v - base_v
    rag_vs_lora = rag_v - lora_v

    lora_arrow = "↑" if lora_gain > 0 else ("↓" if lora_gain < 0 else "→")
    rag_arrow = "↑" if rag_gain > 0 else ("↓" if rag_gain < 0 else "→")
    rvl_arrow = "↑" if rag_vs_lora > 0 else ("↓" if rag_vs_lora < 0 else "→")

    row = f"{metric:<20} {lora_gain:>+9.4f} {lora_arrow} {rag_gain:>+9.4f} {rag_arrow} {rag_vs_lora:>+9.4f} {rvl_arrow}"

    if "LLM_Baseline" in model_results:
        llm_v = model_results["LLM_Baseline"][metric]
        llm_gain = llm_v - base_v
        llm_arrow = "↑" if llm_gain > 0 else ("↓" if llm_gain < 0 else "→")
        row += f" {llm_gain:>+9.4f} {llm_arrow}"

    print(row)

# ========================================
# Print Sample Predictions
# ========================================
print("\n" + "="*60)
print("🔍 Sample Predictions (first 3)")
print("="*60)
for i in range(min(3, len(eval_queries))):
    print(f"\n--- Sample {i+1} ---")
    print(f"  Question: {eval_queries[i][:80]}...")
    print(f"  Reference:  {eval_refs[i][:80]}")
    print(f"  Base Model: {base_preds[i][:80]}")
    print(f"  LoRA Model: {lora_preds[i][:80]}")
    print(f"  RAG:        {rag_preds[i][:80]}")
    if llm_preds is not None:
        print(f"  LLM:        {llm_preds[i][:80]}")

# ========================================
# Visualize results
# ========================================
print("\n" + "="*60)
print("Generating Visualizations")
print("="*60)

visualizer = ResultVisualizer(output_dir=config.outputs.plots_dir)
visualizer.plot_comparison(model_results, save_name="model_comparison_full.png", bertscore_model_label=bertscore_model_resolved)
visualizer.plot_radar_chart(model_results, save_name="radar_comparison.png")

print(f"\n✅ Plots saved to {config.outputs.plots_dir}")

# ========================================
# Save metrics to JSON
# ========================================
import json
metrics_output = {
    "comparison_results": model_results,
    "latency_ms": model_latencies,
    "eval_sample_count": len(eval_queries),
    "best_k": best_k
}
Path(config.outputs.plots_dir).parent.mkdir(parents=True, exist_ok=True)
metrics_path = Path(config.outputs.plots_dir).parent / "comparison_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_output, f, indent=2)
print(f"✅ Metrics saved to {metrics_path}")

print("\n✅ Comprehensive comparison completed!")


STEP 8: Comprehensive Model Comparison

Evaluating 5 samples

[1/4] Generating Base Model predictions...
  ✅ Done | Avg latency: 20406.3 ms/query
[2/4] Generating LoRA Model predictions...
  ✅ Done | Avg latency: 25387.0 ms/query
[3/4] Generating RAG predictions (top_k=2)...


2026-08-27 21:15:58,648 - src.generators.llm_baseline - ERROR - API key not found. Set the 'AQ.Ab8RN6LYk_zAboKW6v9M4NXgIwTOkRH87YL4uZwaOMgzDwptVA' environment variable before initializing LargeLLMGenerator.
2026-08-27 21:15:58,650 - absl - INFO - Using default tokenizer.


  ✅ Done | Avg latency: 33930.7 ms/query
[4/4] Generating LLM Baseline predictions...
  ⚠️ LLM baseline skipped: Gemini client not initialized (no/invalid API key).

Calculating Evaluation Metrics

  Computing Base_Model metrics...


2026-08-27 21:15:59,862 - absl - INFO - Using default tokenizer.


  Computing LoRA_Model metrics...


2026-08-27 21:16:00,782 - absl - INFO - Using default tokenizer.


  Computing RAG_Pipeline metrics...

📊 Model Comparison Results

Metric                   Base_Model     LoRA_Model   RAG_Pipeline
-----------------------------------------------------------------
BLEU                         0.2259         0.1654         0.0044
ROUGE-1                      0.5598         0.5048         0.2413
ROUGE-L                      0.4560         0.4199         0.2329
BERTScore                    0.8759         0.8650         0.7694
Response_Accuracy            0.9520         0.9520         0.6293
-----------------------------------------------------------------
Avg_Latency (ms)            20406.3        25387.0        33930.7

📈 Gain Analysis (vs Base_Model)

Metric                  LoRA Gain     RAG Gain  RAG vs LoRA
-----------------------------------------------------------
BLEU                   -0.0605 ↓   -0.2215 ↓   -0.1610 ↓
ROUGE-1                -0.0550 ↓   -0.3185 ↓   -0.2635 ↓
ROUGE-L                -0.0361 ↓   -0.2231 ↓   -0.1870 ↓
BERTScore       

2026-08-27 21:16:02,373 - src.evaluation.visualization - INFO - ✅ Saved comparison plot to output/plots/model_comparison_full.png
2026-08-27 21:16:02,898 - src.evaluation.visualization - INFO - ✅ Saved radar chart to output/plots/radar_comparison.png



✅ Plots saved to output/plots
✅ Metrics saved to output/comparison_metrics.json

✅ Comprehensive comparison completed!
CPU times: user 6min 26s, sys: 56.9 s, total: 7min 23s
Wall time: 6min 42s


In [14]:
%%time

print("\n" + "="*60)
print("🎉 PIPELINE EXECUTION COMPLETE!")
print("="*60)

# ========================================
# Dataset & Model Summary
# ========================================
print("\n📊 Configuration Summary:")
print(f"  - Dataset: {len(spider_data)} samples (Spider → MongoDB)")
print(f"  - Base Model: {config.model_name}")
print(f"  - Embedding Model: {config.embedding.external_model}")
print(f"  - LoRA: r={config.lora.r}, alpha={config.lora.alpha}, {config.lora.epochs} epoch(s)")
print(f"  - Trainable Params: {config.lora.trainable_fraction*100:.0f}%")
print(f"  - RAG: Best top_k = {best_k}")
print(f"  - Structural Language: {config.retrieval.structural_language}")
print(f"  - Evaluation: {len(eval_samples)} samples")

# ========================================
# Model Comparison Results 
# ========================================
if 'model_results' in dir():
    print("\n" + "-"*60)
    print("📈 Final Model Performance :")
    print("-"*60)
    
    metric_names = list(model_results["Base_Model"].keys())
    model_names = list(model_results.keys())
    
    # Header
    header = f"  {'Metric':<20}"
    for m in model_names:
        header += f" {m:>14}"
    print(header)
    print("  " + "-"*(20 + 15*len(model_names)))
    
    # Rows
    for metric in metric_names:
        row = f"  {metric:<20}"
        for model in model_names:
            val = model_results[model].get(metric, 0.0)
            row += f" {val:>14.4f}"
        print(row)
    
    # Find best model by BLEU
    best_model = max(model_names, key=lambda m: model_results[m].get("BLEU", 0.0))
    best_bleu = model_results[best_model]["BLEU"]
    print(f"\n  🏆 Best Model (by BLEU): {best_model} ({best_bleu:.4f})")

# ========================================
# Latency Summary
# ========================================
if 'model_latencies' in dir():
    print("\n" + "-"*60)
    print("⚡ Average Generation Latency:")
    print("-"*60)
    for model_name, latency in model_latencies.items():
        if latency is not None:
            print(f"  - {model_name:<20}: {latency:>10.1f} ms/query")

# ========================================
# RAG Improvement Summary 
# ========================================
if 'rag_improvement' in dir():
    print("\n" + "-"*60)
    print("🔍 RAG Improvement over LoRA Baseline:")
    print("-"*60)
    for metric, gain in rag_improvement.get('gain_over_baseline', {}).items():
        arrow = "↑" if gain > 0 else ("↓" if gain < 0 else "→")
        print(f"  - {metric:<25}: {gain:>+8.4f} {arrow}")

# ========================================
# Output Artifacts
# ========================================
print("\n" + "-"*60)
print("📁 Output Artifacts:")
print("-"*60)
print(f"  - LoRA Adapter:    {config.lora.output_dir}")
print(f"  - Indices:         data/indices/")
print(f"  - Plots:           {config.outputs.plots_dir}/")
_actual_metrics_path = Path(config.outputs.plots_dir).parent / "comparison_metrics.json"
print(f"  - Metrics JSON:    {_actual_metrics_path}")

# List generated plot files
plots_dir = Path(config.outputs.plots_dir)
if plots_dir.exists():
    plot_files = list(plots_dir.glob("*.png"))
    if plot_files:
        print(f"  - Generated Plots ({len(plot_files)}):")
        for pf in plot_files:
            print(f"      • {pf.name}")

# ========================================
# Execution Time
# ========================================
print("\n" + "="*60)
print("✅ All pipeline tasks completed successfully!")
print("="*60)


🎉 PIPELINE EXECUTION COMPLETE!

📊 Configuration Summary:
  - Dataset: 1114 samples (Spider → MongoDB)
  - Base Model: Qwen/Qwen2.5-Coder-0.5B-Instruct
  - Embedding Model: BAAI/bge-small-en-v1.5
  - LoRA: r=16, alpha=32, 1 epoch(s)
  - Trainable Params: 100%
  - RAG: Best top_k = 2
  - Structural Language: mongodb
  - Evaluation: 5 samples

------------------------------------------------------------
📈 Final Model Performance :
------------------------------------------------------------
  Metric                   Base_Model     LoRA_Model   RAG_Pipeline
  -----------------------------------------------------------------
  BLEU                         0.2259         0.1654         0.0044
  ROUGE-1                      0.5598         0.5048         0.2413
  ROUGE-L                      0.4560         0.4199         0.2329
  BERTScore                    0.8759         0.8650         0.7694
  Response_Accuracy            0.9520         0.9520         0.6293

  🏆 Best Model (by BLEU): Bas